In [ ]:
# Import the Roman HGA pointing helpers and create one reusable model instance
import RST_specific_functions as RST

model = RST.RomanHGAPointingModel()


In [ ]:
# Define yaw / pitch / roll attitude grids.
# Old two-value inputs still work, but tuples below use the full (yaw, pitch, roll) convention.
def make_attitude_grid(yaws=(0,), pitches=(36, 0, -36), rolls=(-15, 0, 15)):
    return [(yaw, pitch, roll) for yaw in yaws for pitch in pitches for roll in rolls]

all_for_attitudes = make_attitude_grid()
more_for_attitudes = make_attitude_grid(
    yaws=(0,),
    pitches=(36, 30, 24, 18, 12, 6, 0, -6, -12, -18, -24, -30, -36),
    rolls=(-15, -10, -5, 0, 5, 10, 15),
)

# OS11 is a time-ordered operating scenario rather than a static FOR sampling.
os11_raw_text = '''125.52325,-4.576647,0,0
125.52325,-4.576647,0,432001
-149.338,-8.2617,10.2548,433801
-149.6054,-9.2828,12.0064,614700
-58.7289,-6.3305,-12.4764,616500
-58.7248,-6.3903,-12.5129,622801
-58.7242,-6.3988,13.4819,623701
-58.7202,-6.4586,13.4453,630001
-58.7196,-6.4672,-12.5599,630901
-58.7154,-6.5269,-12.5965,637201
-58.7148,-6.5355,13.3983,638101
-58.7107,-6.5953,13.3617,644401
-149.6552,-9.4597,-13.6878,646201
-149.6595,-9.4748,-13.6615,648901
-58.7065,-6.655,13.3251,650701
-58.7022,-6.7148,13.2885,657000
-58.7016,-6.7233,-12.7167,657900
-58.6973,-6.7831,-12.7533,664200
-58.6967,-6.7916,13.2415,665100
-58.6923,-6.8514,13.2049,671400
-58.6917,-6.8599,-12.8004,672300
-58.6873,-6.9197,-12.837,678600
-149.7104,-9.6514,12.6446,680400
-149.7444,-9.7673,12.846,701102
-58.67,-7.1502,-12.9783,702902
-58.6655,-7.2099,-13.015,709201
-58.6648,-7.2184,12.9798,710101
-58.6602,-7.2782,12.9431,716401
-58.6595,-7.2867,-13.0621,717301
-58.6549,-7.3464,-13.0988,723601
-58.6542,-7.3549,12.896,724501
-58.6495,-7.4147,12.8593,730801
-149.7969,-9.9434,-12.8474,732601
-149.8015,-9.9584,-12.8211,735302
-58.6447,-7.4744,12.8226,737102
-58.6399,-7.5341,12.7859,743401
-58.6392,-7.5426,-13.2194,744301
-58.6344,-7.6023,-13.2561,750601
-58.6337,-7.6109,12.7387,751501
-58.6288,-7.6706,12.7019,757801
-58.6281,-7.6791,-13.3033,758701
-58.6232,-7.7388,-13.3401,765001
-149.855,-10.1342,13.4859,766801
-149.8908,-10.2495,13.6878,787503
-58.6039,-7.969,-13.4819,789303'''
os11_attitudes = RST.parse_os11_attitudes(os11_raw_text)

# Example yaw sweep you can turn on when you want yaw included in the trade space.
yaw_sweep_attitudes = make_attitude_grid(
    yaws=(-10, 0, 10),
    pitches=(36, 0, -36),
    rolls=(-15, 0, 15),
)


In [ ]:
# Keep the notebook interface light: define a target once, then solve attitudes with one helper.
def print_hga_inputs(target_vector, attitudes, initial_guess=(0, 0)):
    print('HGA gimbal inputs for target vector:')
    print(target_vector)
    print()

    results = model.solve_across_attitudes(target_vector, attitudes, initial_guess=initial_guess)
    for result in results:
        attitude = result.attitude
        gimbal = result.gimbal_angles
        if attitude.time is None:
            attitude_text = f'Pitch={attitude.pitch}, Roll={attitude.roll}, Yaw={attitude.yaw}'
        else:
            attitude_text = f'Time={attitude.time}, Pitch={attitude.pitch}, Roll={attitude.roll}, Yaw={attitude.yaw}'

        print(
            f'{attitude_text}: '
            f'y_track={gimbal.y_track:.6f}, '
            f'x_track={gimbal.x_track:.6f}, '
            f'error={result.pointing_error:.6e}, '
            f'success={result.success}'
        )

    return results


def print_pitch_roll_thermal_desktop_exports(results, x_symbol_name, y_symbol_name):
    exports = model.format_thermal_desktop_gimbal_exports(results)
    print(x_symbol_name)
    print(exports.x_track)
    print('')
    print(y_symbol_name)
    print(exports.y_track)
    print('')
    return exports


def print_time_thermal_desktop_exports(results, x_symbol_name, y_symbol_name, start_time=None, final_else='11'):
    exports = model.format_thermal_desktop_time_gimbal_exports(
        results,
        start_time=start_time,
        final_else=final_else,
    )
    print(x_symbol_name)
    print(exports.x_track)
    print('')
    print(y_symbol_name)
    print(exports.y_track)
    print('')
    return exports


In [ ]:
# Choose the pointing variation and the attitude set you want to solve.
reference_cases = {
    'wide': {
        'name': 'Wide configuration',
        'initial_gimbal': (10.013536790487828, -0.4368647026044624),
        'target_attitude': (0, 0, 0),
        'thermal_desktop_symbols': {
            'for_x': 'STOP_HG_Wide_Xtrack',
            'for_y': 'STOP_HG_Wide_Ytrack',
            'os11_x': 'STOP_HG_OS11_Wide_Xtrack',
            'os11_y': 'STOP_HG_OS11_Wide_Ytrack',
        },
    },
    'skinny': {
        'name': 'Skinny configuration',
        'initial_gimbal': (10, 15),
        'target_attitude': (0, 36, -15),
        'thermal_desktop_symbols': {
            'for_x': 'STOP_HG_Skinny_Xtrack',
            'for_y': 'STOP_HG_Skinny_Ytrack',
            'os11_x': 'STOP_HG_OS11_Skinny_Xtrack',
            'os11_y': 'STOP_HG_OS11_Skinny_Ytrack',
        },
    },
    'shady': {
        'name': 'Shady configuration',
        'initial_gimbal': (-26, 0),
        'target_attitude': (0, 0, 0),
        'thermal_desktop_symbols': {
            'for_x': 'STOP_HG_Shady_Xtrack',
            'for_y': 'STOP_HG_Shady_Ytrack',
            'os11_x': 'STOP_HG_OS11_Xtrack',
            'os11_y': 'STOP_HG_OS11_Ytrack',
        },
    },
}

attitude_sets = {
    'for': {
        'name': 'FOR sweep',
        'attitudes': more_for_attitudes,
        'thermal_desktop_mode': 'pitch_roll',
    },
    'os11': {
        'name': 'OS11 operating scenario',
        'attitudes': os11_attitudes,
        'thermal_desktop_mode': 'time',
        'thermal_desktop_start_time': 612000.0,
        'thermal_desktop_final_else': '11',
    },
}

selected_variation = 'shady'   # Change to 'wide', 'skinny', or 'shady'.
selected_attitude_type = 'os11'  # Change to 'for' or 'os11'.

case = reference_cases[selected_variation]
attitude_set = attitude_sets[selected_attitude_type]

print(case['name'])
print(attitude_set['name'])
print('')

target_vector = model.define_target(case['initial_gimbal'], case['target_attitude'])
results = print_hga_inputs(target_vector, attitude_set['attitudes'])

if attitude_set['thermal_desktop_mode'] == 'pitch_roll':
    print_pitch_roll_thermal_desktop_exports(
        results,
        case['thermal_desktop_symbols']['for_x'],
        case['thermal_desktop_symbols']['for_y'],
    )
else:
    print_time_thermal_desktop_exports(
        results,
        case['thermal_desktop_symbols']['os11_x'],
        case['thermal_desktop_symbols']['os11_y'],
        start_time=attitude_set['thermal_desktop_start_time'],
        final_else=attitude_set['thermal_desktop_final_else'],
    )


In [ ]:
# Optional example: include yaw in the target definition and in a small yaw sweep.
# The OS11 export path already supports timed yaw/pitch/roll attitudes.
run_yaw_example = False

if run_yaw_example:
    yaw_target_vector = model.define_target((10.013536790487828, -0.4368647026044624), (10, 12, -5), verbose=False)
    print_hga_inputs(yaw_target_vector, yaw_sweep_attitudes)
